In [ ]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [2]:
labeled_df = pd.read_csv('../data/labeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)
unlabeled_df = pd.read_csv('../data/unlabeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)

print(f"labeled_df shape: {labeled_df.shape}")
print(f"unlabeled_df shape: {unlabeled_df.shape}")

labeled_df shape: (25000, 3)
unlabeled_df shape: (50000, 2)


In [3]:
import my_utils # noqa: F401

wordlist_labeled_train = labeled_df.review.nlp.review_to_wordlist()
wordlist_unlabeled_train = unlabeled_df.review.nlp.review_to_wordlist()

print(f"wordlist_labeled_train shape: {wordlist_labeled_train.shape}")
print(f"wordlist_unlabeled_train shape: {wordlist_unlabeled_train.shape}")

wordlist_labeled_train shape: (25000,)
wordlist_unlabeled_train shape: (50000,)


### TF‑IDF representation  

Convert the corpus into a sparse TF‑IDF matrix.

max_df=0.85 removes extremely common words.

English stopwords are removed automatically.

In [4]:
wordlist_all =  pd.concat([wordlist_labeled_train, wordlist_unlabeled_train], ignore_index=True)
corpus_all = [" ".join(words) for words in wordlist_all]
corpus_train = [" ".join(words) for words in wordlist_labeled_train]

print(f"Total number of reviews in corpus_all: {len(corpus_all)}")
print(f"Total number of reviews in corpus_train: {len(corpus_train)}")

Total number of reviews in corpus_all: 75000
Total number of reviews in corpus_train: 25000


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english', max_df=0.85)
vectorizer = vectorizer.fit(corpus_all)
tfidf_matrix = vectorizer.transform(corpus_train)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

Vocabulary size: 123129
TF-IDF matrix shape: (25000, 123129)


In [6]:
from sklearn.decomposition import TruncatedSVD

n_components=200

svd = TruncatedSVD(n_components=n_components, random_state=42)
X_reduced = svd.fit_transform(tfidf_matrix)
X_reduced.shape

(25000, 200)

### Baseline sentiment models  

Split the reduced TF‑IDF features into train/test sets to evaluate simple classifiers.

This establishes a baseline before experimenting with more advanced embeddings.

In [7]:
from sklearn.model_selection import train_test_split


X_train_sentiment, X_test_sentiment, y_train_sentiment, y_test_sentiment = train_test_split(
    X_reduced, labeled_df.sentiment, test_size=0.2, random_state=42
)

### Baseline models  

Evaluate several linear and tree‑based classifiers on the reduced TF‑IDF features.

Logistic Regression and Linear SVM are strong baselines for text classification.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

evaluators = [
    (LogisticRegression(max_iter=500), 'LogisticRegression'),
    (LinearSVC(), 'LinearSVC'),
    (RandomForestClassifier(random_state=42, max_depth=10), 'RandomForestClassifier')    
]


In [9]:
for tup in evaluators:
    (clf, eval_class) = tup
    
    print(f'Evaluating {eval_class}')
    clf.fit(X_train_sentiment, y_train_sentiment)
    score = clf.score(X_test_sentiment, y_test_sentiment)    
    print(f'{eval_class} sentiment score: {score}')
    print("\n")

Evaluating LogisticRegression
LogisticRegression sentiment score: 0.8664


Evaluating LinearSVC
LinearSVC sentiment score: 0.8688


Evaluating RandomForestClassifier
RandomForestClassifier sentiment score: 0.8088




### Summary  

Baseline classifiers trained on reduced TF‑IDF features provide a reference point for future iterations.

Next steps include experimenting with Word2Vec embeddings and improving preprocessing.